# Section 5: input steering length and prompt baseline

Fixed replay, layer 17 / head 0, prediction step 128. All points and the prompt baseline use the same 100 behaviors with at least 64 input tokens. Shaded regions are 95% bootstrap confidence intervals across behaviors.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
summary = json.loads((ROOT / "results/section5_fixed_summary.json").read_text())
series = sorted(
    (row for row in summary["input_length_matched_cohort"] if row["alpha"] == 1),
    key=lambda row: row["k"] if row["k"] > 0 else float("inf"),
)
assert [row["k"] for row in series] == [1, 4, 16, 64, -1]
assert {row["n"] for row in series} == {100}

cohort = []
for path in sorted((ROOT / "results").glob("section5_fixed_[0-9][0-9][0-9].json")):
    state = json.loads(path.read_text())
    unit = state["units"]["17_0"]
    if unit["base_length"] >= 64:
        prompt = next(row for row in unit["rows"] if row["id"] == "prompt_m8")
        assert prompt["eligible"]
        cohort.append(prompt)
assert len(cohort) == 100

baseline = {}
for field in ("R", "C"):
    values = np.array([row[field] for row in cohort if row[field] is not None])
    assert len(values) == 100
    draws = np.random.default_rng(42).integers(0, len(values), size=(2000, len(values)))
    lower, upper = np.quantile(values[draws].mean(axis=1), [.025, .975])
    baseline[field] = {"mean": float(values.mean()), "lower": float(lower), "upper": float(upper)}

baseline

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
x = np.arange(len(series))
labels = [str(row["k"]) if row["k"] > 0 else "Full" for row in series]
for ax, field in zip(axes, ("R", "C")):
    mean = [row[field]["mean"] for row in series]
    lower = [row[field]["lower"] for row in series]
    upper = [row[field]["upper"] for row in series]
    ax.plot(x, mean, "o-", color="tab:blue", label="Steering")
    ax.fill_between(x, lower, upper, color="tab:blue", alpha=.15)
    ax.axhline(baseline[field]["mean"], color="tab:green", linestyle="--", label="Prompt (m=8)")
    ax.axhspan(baseline[field]["lower"], baseline[field]["upper"], color="tab:green", alpha=.12)
    ax.set_xticks(x, labels)
    ax.set_xlabel("Steered input tokens (m=8, α=1)")
    ax.set_ylabel(field)
    ax.grid(alpha=.2)
    ax.legend(fontsize=8)
axes[0].set_ylim(top=1.015)
fig.suptitle("Matched cohort: n=100")
output = ROOT / "figs/section5_input_length.pdf"
fig.savefig(output)
plt.show()